# trainer-subclass-extend — ex2: three-level trainer chain — each subclass extends _step via super() and adds one metric

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `trainer-subclass-extend`. Running the final beacon cell reports progress against the `Trainer: subclass extend pattern` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Trainer: subclass extend pattern` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`trainer-subclass-extend`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "trainer-subclass-extend"
DD_SUBTOPIC = "Trainer: subclass extend pattern"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## 3-level subclass chain — `super()._step` MRO walking

Ex1 had ONE subclass extending ONE base. The deepening move stacks a THIRD level: `BaseTrainer ← LoggingTrainer ← FrozenLoggingTrainer`. Each level adds its own metric by calling `super()._step(batch)` first and then mutating the returned dict.

```python
class BaseTrainer:
    def _step(self, batch):
        return {'loss': float(batch[0].abs().sum())}

class LoggingTrainer(BaseTrainer):
    def _step(self, batch):
        d = super()._step(batch)
        d['input_mag'] = float(batch[0].abs().mean())
        return d

class FrozenLoggingTrainer(LoggingTrainer):
    def _step(self, batch):
        d = super()._step(batch)  # walks MRO → LoggingTrainer._step
        d['grad_norm'] = 0.0       # head is frozen this run
        return d
```

**Why MRO matters.** `super()._step(batch)` does NOT mean `BaseTrainer._step(batch)`. Python walks the method resolution order (`type(self).__mro__`) and calls the NEXT class in the chain. From `FrozenLoggingTrainer.__mro__`, the next class is `LoggingTrainer`, which itself defers to `BaseTrainer` via its own `super()`. The chain composes — each level's metric ends up in the final dict.

### Exercise 2 — three-level trainer chain — each subclass extends _step via super() and adds one metric

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `super()._step(batch)` at each of three subclass levels so Python's MRO walks the chain Base → Logging → FrozenLogging and the final dict accumulates the metrics added by every level.
> Keywords: mro, super, subclass, trainer, chain
> ```

**KCs targeted:** `super-walks-mro-not-base`, `each-level-extends-dict`

`BaseTrainer` is provided in the stub. Its `_step(batch)` returns `{'loss': float(batch[0].abs().sum())}`. Build TWO further subclass levels:

1. `LoggingTrainer(BaseTrainer)`:
   - Override `_step(self, batch)`:
     a. Call `super()._step(batch)` and capture the dict `d`.
     b. Add `d['input_mag'] = float(batch[0].abs().mean())`.
     c. Return `d`.
2. `FrozenLoggingTrainer(LoggingTrainer)`:
   - Override `_step(self, batch)`:
     a. Call `super()._step(batch)` (walks MRO to `LoggingTrainer._step`, which itself defers to `BaseTrainer._step`).
     b. Add `d['grad_norm'] = 0.0` (head is frozen, no grad).
     c. Return `d`.

Return both classes from `ex2_trainer_chain()` as a 2-tuple `(LoggingTrainer, FrozenLoggingTrainer)`.

Constraints:
- Each subclass MUST call `super()._step(batch)` — do not re-implement the base body.
- `FrozenLoggingTrainer._step` MUST NOT call `LoggingTrainer._step` directly by name — use `super()`.
- The final dict from `FrozenLoggingTrainer._step` must contain all three keys: `'loss'`, `'input_mag'`, `'grad_norm'`.

In [ ]:
class LoggingTrainer(BaseTrainer):
    def _step(self, batch):
        d = super()._step(batch)
        d['input_mag'] = float(batch[0].abs().mean())
        return d

class FrozenLoggingTrainer(LoggingTrainer):
    def _step(self, batch):
        d = super()._step(batch)
        d['grad_norm'] = 0.0
        return d

def ex2_trainer_chain():
    return (LoggingTrainer, FrozenLoggingTrainer)


<details><summary>Solution</summary>

```python
class LoggingTrainer(BaseTrainer):
    def _step(self, batch):
        d = super()._step(batch)
        d['input_mag'] = float(batch[0].abs().mean())
        return d

class FrozenLoggingTrainer(LoggingTrainer):
    def _step(self, batch):
        d = super()._step(batch)
        d['grad_norm'] = 0.0
        return d

def ex2_trainer_chain():
    return (LoggingTrainer, FrozenLoggingTrainer)
```

**`super()` walks the MRO, not the static base.** From `FrozenLoggingTrainer._step`, `super()._step(batch)` does NOT mean `BaseTrainer._step(batch)`. It means 'the next class in `type(self).__mro__` after `FrozenLoggingTrainer`' — which is `LoggingTrainer`. The chain composes automatically.

**Why explicit naming breaks the chain.** If `FrozenLoggingTrainer._step` called `LoggingTrainer._step(self, batch)` directly, it would still work for this exact hierarchy. But it breaks under multiple inheritance — a sibling subclass wouldn't get a chance to run. `super()` is the future-proof form.

**Same dict mutated in place.** Each level adds its key into the dict returned by `super()._step`. No copying. The base creates the dict; each subclass extends it. By the time control returns to the caller, all three keys are present.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()